# 06 Report Outputs

This notebook creates and displays the final report figures for the full project story: prepared incident-window data, anomaly-detection model training, timestamp and incident-level evaluation, alarm episode grouping, full all-episode XAI evidence, and operator-facing triage.

It does not retrain any model and does not change predictions, thresholds, metrics, or saved model artifacts. It only loads final project outputs and turns them into clean figures, tables, and captions suitable for the written report. The generated figures are shown directly in this notebook and also saved as high-resolution PNG files. The intended framing is that XAI is an evidence-quality layer: it explains model behavior, not the physical cause of traffic conditions.


## Imports and Paths

The reusable figure-generation code lives in `traffic_incident_xai.report_outputs`. The notebook calls that module so the report figures are displayed inside the notebook and can also be regenerated consistently without duplicating long plotting code across cells.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "traffic_incident_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "traffic_incident_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))

from traffic_incident_xai import config
from traffic_incident_xai import report_outputs as report

report.REPORT_FIG_DIR


## Helper Functions for Clean Labels

The report helper module converts technical names into reader-friendly labels. For example, `speed_smoothed_t_minus_1` becomes Speed, 5 min before, and `lime_style_local_surrogate` becomes LIME-style local surrogate.


In [ ]:
examples = [
    "speed_smoothed",
    "occ_smoothed",
    "speed_smoothed_t_minus_1",
    "speed_smoothed_t_minus_12",
    "occ_smoothed_t_minus_4",
]
for value in examples:
    print(f"{value}  ->  {report.feature_label(value)}")

for value in ["treeshap", "deepshap", "sequence_deepshap", "lime_style_local_surrogate"]:
    print(f"{value}  ->  {report.method_label(value)}")


## Load Final Outputs

The notebook uses final outputs only: the processed incident-window dataset, final predictions and thresholds, final model metrics, alarm episode tables, and all-episode XAI evidence tables.


In [ ]:
ctx = report.load_context()
summary = pd.DataFrame([
    {"Output": "Processed dataset rows", "Count": len(ctx.training)},
    {"Output": "Unique streams", "Count": ctx.training["stream_id"].nunique()},
    {"Output": "Final prediction rows", "Count": len(ctx.predictions)},
    {"Output": "Alarm episodes", "Count": len(ctx.alarm_episodes)},
    {"Output": "XAI alarm representative targets", "Count": len(ctx.xai_targets)},
    {"Output": "XAI explained model-method cases", "Count": len(ctx.xai_evidence)},
    {"Output": "Local XAI attribution rows", "Count": ctx.xai_local_row_count},
])
summary


**Interpretation:** this confirms that the notebook is using the full final outputs, including the all-alarm XAI tables across all five folds.


## Project and Data Protocol Figures

These figures establish the experimental design: the end-to-end workflow, the 240-hour/12-hour incident-window structure, the stream-level cross-validation split, and the label/window distribution.


In [ ]:
fig = report.project_pipeline_overview(ctx)
fig


![Report figure: Project pipeline overview](../results/figures/report/project_pipeline_overview.png)


**Interpretation:** the project workflow moves from prepared incident files to alarm episodes and XAI-supported triage, so the contribution is larger than raw model scoring.


In [ ]:
fig = report.incident_window_protocol(ctx)
fig


![Report figure: Incident-window protocol](../results/figures/report/incident_window_protocol.png)


**Interpretation:** each selected stream has a normal baseline period followed by a controlled test window containing the target incident period.


In [ ]:
fig = report.stream_level_cv_diagram(ctx)
fig


![Report figure: Stream-level cross-validation diagram](../results/figures/report/stream_level_cv_diagram.png)


**Interpretation:** whole streams stay together during cross-validation. This avoids mixing temporally related rows from the same stream across train and test.


In [ ]:
fig = report.dataset_label_window_summary(ctx)
fig


![Report figure: Dataset label and window summary](../results/figures/report/dataset_label_window_summary.png)


**Interpretation:** incident timestamps are a small part of the test windows, so accuracy alone is not a useful summary. The report should emphasize recall, incident detection, and alarm episode behavior.


## Model Evaluation Figures

These figures compare model behavior at incident level and timestamp level, then check fold-level stability and threshold behavior.


In [ ]:
figs = report.model_performance_figures(ctx)
figs[0]


![Report figure: Incident-level model comparison](../results/figures/report/incident_level_model_comparison.png)


In [ ]:
figs[1]


![Report figure: Timestamp-level model comparison](../results/figures/report/timestamp_level_model_comparison.png)


**Interpretation:** incident-level metrics answer whether the event was detected, while timestamp-level metrics describe row-by-row prediction behavior.


In [ ]:
fig = report.fold_performance_stability(ctx)
fig


![Report figure: Fold performance stability](../results/figures/report/fold_performance_stability.png)


**Interpretation:** fold stability shows whether performance depends heavily on one held-out stream group.


In [ ]:
fig = report.threshold_score_overview(ctx)
fig


![Report figure: Threshold and score overview](../results/figures/report/threshold_score_overview.png)


**Interpretation:** score margin above threshold is a triage signal: a barely positive alarm and a far-above-threshold alarm should not be described with the same confidence.


## Alarm Episode Figures

The next figures show how timestamp predictions become operational alarm episodes.


In [ ]:
fig = report.example_alarm_timeline(ctx)
fig


![Report figure: Example alarm timeline](../results/figures/report/example_alarm_timeline.png)


**Interpretation:** this example connects alarms to the actual model inputs: smoothed speed and occupancy over the test window.


In [ ]:
fig = report.timestamp_to_alarm_episode_grouping(ctx)
fig


![Report figure: Timestamp predictions grouped into alarm episode](../results/figures/report/timestamp_to_alarm_episode_grouping.png)


**Interpretation:** consecutive positive timestamps are grouped into one alarm episode, which is the correct unit for operator review.


In [ ]:
fig = report.alarm_episode_summary(ctx)
fig


![Report figure: Alarm episode summary](../results/figures/report/alarm_episode_summary.png)


**Interpretation:** episode counts, durations, and model agreement describe operator workload and alarm strength more clearly than timestamp counts alone.


## XAI Evidence Figures

These figures use XAI computed for every alarm episode representative row across the five folds. XAI explains model behavior, not physical causality.


In [ ]:
fig = report.xai_coverage_overview(ctx)
fig


![Report figure: XAI coverage overview](../results/figures/report/xai_coverage_overview.png)


**Interpretation:** XAI coverage is complete for the planned alarm representative rows, so the evidence summaries describe the alarm set rather than isolated examples.


## Where XAI Values Come From

Before interpreting SHAP/LIME-style diagrams, the report needs to clarify what an attribution value means. The next figure shows the path from one alarm representative row to local attribution values and then to operator-facing evidence metrics.


In [ ]:
fig = report.shapley_value_context(ctx)
fig


![Report figure: SHAP/LIME attribution value context](../results/figures/report/shapley_value_context.png)


**Interpretation:** an attribution value is not a traffic measurement. It is a model-explanation value that describes how much a lagged input contributed to the model output for one alarm row, relative to reference behaviour.


## General SHAP-Style Influence Summary

The next figure is the generalized version of the familiar SHAP beeswarm plot. Instead of showing one alarm, each dot represents one explained alarm episode representative row. Because the figure combines tree, MLP, and LSTM SHAP-style methods, the horizontal axis uses normalized signed explanation share rather than raw SHAP units.


In [ ]:
fig = report.shap_style_summary_beeswarm(ctx)
fig


![Report figure: SHAP-style summary beeswarm](../results/figures/report/shap_style_summary_beeswarm_all_episodes.png)


**Interpretation:** this figure supports the conclusion-level XAI discussion. Features near the top are the lagged Speed/Occupancy inputs that most often carry explanation weight across alarm episodes. Dots to the right support the alarm output; dots to the left oppose it. The colour shows whether the underlying traffic feature value was low or high within that feature range.


In [ ]:
fig = report.local_treeshap_value_example(ctx)
fig


![Report figure: Local TreeSHAP attribution example](../results/figures/report/local_treeshap_value_example.png)


**Interpretation:** this is a real local TreeSHAP example. The bars show which lagged Speed/Occupancy inputs contributed most to one tree-model alarm. Positive or negative direction describes model-output direction, not physical causality.


In [ ]:
fig = report.temporal_attribution_heatmap(ctx)
fig


![Report figure: Temporal attribution heatmap](../results/figures/report/temporal_attribution_heatmap_all_episodes.png)


**Interpretation:** recent traffic evidence shows whether models are relying mainly on the last 0-15 minutes before the alarm or on older traffic context.


In [ ]:
fig = report.feature_family_share(ctx)
fig


![Report figure: Feature-family attribution share](../results/figures/report/feature_family_share_by_model_method.png)


**Interpretation:** speed and occupancy shares show which input family dominates, but temporal structure is more informative because only two input variables are used.


In [ ]:
fig = report.explanation_concentration(ctx)
fig


![Report figure: Explanation concentration](../results/figures/report/explanation_concentration_by_model_method.png)


**Interpretation:** concentrated explanations are easier to turn into operator-facing evidence statements. Diffuse explanations should be worded more cautiously.


In [ ]:
fig = report.score_margin_vs_concentration(ctx)
fig


![Report figure: Score margin versus explanation concentration](../results/figures/report/score_margin_vs_explanation_concentration.png)


**Interpretation:** alarm strength and explanation concentration are different dimensions. The triage layer benefits from using both.


In [ ]:
fig = report.shap_lime_agreement(ctx)
fig


![Report figure: SHAP and LIME-style agreement](../results/figures/report/shap_lime_agreement_by_model.png)


**Interpretation:** SHAP/LIME-style agreement indicates whether two explanation methods emphasize similar lagged evidence for the same alarm case.


## SHAP vs LIME Case Profiles

Aggregate recent-attribution percentages are useful, but they can hide what happens inside individual alarms. This section selects one true incident alarm and one offline false alarm where both TreeSHAP and LIME-style explanations are available. For each case, the figure shows how explanation weight is distributed across the 60-minute lookback window.

The goal is not to prove which method is correct. The goal is to show whether the two XAI methods tell a similar temporal story for the same alarm, or whether one method concentrates evidence in the most recent timestamps while the other spreads evidence over a wider window.


In [ ]:
import matplotlib.pyplot as plt

local = pd.read_csv(config.TABLES_DIR / "xai_local_explanations_all_alarm_episodes.csv")
targets = pd.read_csv(config.TABLES_DIR / "xai_alarm_episode_targets.csv")
methods = ["treeshap", "lime_style_local_surrogate"]

paired = (
    local[local["method"].isin(methods)]
    .groupby(["case_id", "row_id", "model", "fold"])["method"]
    .nunique()
    .reset_index(name="method_count")
)
paired = paired[paired["method_count"] == 2]
meta = paired.merge(
    targets[[
        "representative_row_id", "model", "fold", "stream_id", "score", "threshold",
        "score_margin", "model_agreement_count", "true_label", "case_type",
        "alarm_start", "alarm_end"
    ]],
    left_on=["row_id", "model", "fold"],
    right_on=["representative_row_id", "model", "fold"],
    how="left",
)
meta = meta[meta["model"].isin(["random_forest", "xgboost"])].copy()

true_case = meta[meta["true_label"] == 1].sort_values(
    ["model_agreement_count", "score_margin"], ascending=[False, False]
).iloc[0]
false_case = meta[meta["true_label"] == 0].sort_values(
    ["model_agreement_count", "score_margin"], ascending=[False, False]
).iloc[0]

selected_cases = pd.DataFrame([true_case, false_case])
selected_cases.to_csv(config.TABLES_DIR / "shap_lime_true_false_case_selection.csv", index=False)
selected_cases[["case_id", "model", "fold", "stream_id", "row_id", "true_label", "case_type", "score_margin", "model_agreement_count"]]


In [ ]:
def attribution_profile(case):
    rows = local[(local["case_id"] == case["case_id"]) & local["method"].isin(methods)].copy()
    rows["method_label"] = rows["method"].map({
        "treeshap": "TreeSHAP",
        "lime_style_local_surrogate": "LIME-style",
    })
    rows["method_total"] = rows.groupby("method")["abs_attribution"].transform("sum")
    rows = rows[rows["method_total"] > 0].copy()
    rows["abs_share"] = rows["abs_attribution"] / rows["method_total"]
    return (
        rows.groupby(["method_label", "minutes_before_alarm"], as_index=False)["abs_share"]
        .sum()
        .sort_values("minutes_before_alarm")
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
case_info = [("True incident alarm", true_case), ("Offline false alarm", false_case)]
colors = {"TreeSHAP": "#2C7FB8", "LIME-style": "#F16913"}

for ax, (title, case) in zip(axes, case_info):
    profile = attribution_profile(case)
    for method, group in profile.groupby("method_label"):
        ax.plot(
            group["minutes_before_alarm"],
            group["abs_share"],
            marker="o", linewidth=2.4, markersize=5.5,
            label=method, color=colors[method],
        )
    ax.axvspan(0, 15, color="#E5F5E0", alpha=0.8, label="Recent 0-15 min" if ax is axes[0] else None)
    ax.axvspan(30, 60, color="#FEE6CE", alpha=0.45, label="Older 30-60 min" if ax is axes[0] else None)
    ax.set_title(
        f"{title}\n{report.model_label(case['model'])}, fold {int(case['fold'])}, stream {int(case['stream_id'])}",
        fontsize=11,
    )
    ax.set_xlabel("Minutes before representative alarm row")
    ax.set_xticks([5, 15, 30, 45, 60])
    ax.grid(axis="y", alpha=0.25)
    ax.text(
        0.02, 0.96,
        f"score margin={case['score_margin']:.3f}\nmodels alarming={int(case['model_agreement_count'])}/4",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#BDBDBD"},
    )

axes[0].set_ylabel("Share of absolute local attribution")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("SHAP vs LIME temporal attribution profiles for two alarm episodes", fontsize=14, y=1.02)
fig.text(
    0.5, -0.08,
    "Each line sums normalized absolute attribution at each time lag. Similar line shapes mean SHAP and LIME emphasize similar temporal evidence; different shapes indicate explanation-method divergence.",
    ha="center", va="top", fontsize=9, color="#525252",
)
fig.tight_layout(rect=[0, 0.08, 1, 0.98])
profile_path = report.REPORT_FIG_DIR / "shap_lime_true_false_temporal_profiles.png"
fig.savefig(profile_path, dpi=300, bbox_inches="tight", facecolor="white")
fig


![Report figure: SHAP vs LIME true and false alarm temporal profiles](../results/figures/report/shap_lime_true_false_temporal_profiles.png)


**Interpretation:** this case-study figure makes the SHAP/LIME comparison concrete. Both methods explain the same alarm row, but their temporal attribution profiles can differ. TreeSHAP may concentrate most of the explanation in the most recent lag, while LIME can spread attribution across later parts of the 60-minute window. In the project, this difference is treated as an explanation-stability signal: agreement gives a more stable evidence story, while divergence means the alarm should be interpreted with more caution.


In [ ]:
figs = report.cross_model_agreement_figures(ctx)
figs[0]


![Report figure: Cross-model agreement heatmap](../results/figures/report/cross_model_agreement_heatmap.png)


In [ ]:
figs[1]


![Report figure: Cross-model agreement level summary](../results/figures/report/cross_model_agreement_level_summary.png)


**Interpretation:** cross-model agreement supports the research question by showing whether different model families explain alarms using similar traffic evidence.


## Operator-Facing Triage Figures

These final figures connect alarm score, threshold margin, model agreement, XAI recentness, explanation concentration, and method agreement into operator-support outputs.


In [ ]:
fig = report.strong_vs_borderline_profiles(ctx)
fig


![Report figure: Strong versus borderline alarm profiles](../results/figures/report/strong_vs_borderline_alarm_profiles.png)


**Interpretation:** the strong profile is review-worthy because multiple evidence-quality indicators align. The borderline profile is still visible, but should be treated with lower priority unless external context confirms worsening traffic conditions.


In [ ]:
fig = report.alarm_evidence_profile_summary(ctx)
fig


![Report figure: Alarm evidence profile summary](../results/figures/report/alarm_evidence_profile_summary.png)


**Interpretation:** the final alarm evidence profile summarizes the triage logic: score strength and model agreement show how strongly the system alarmed, while XAI shows whether evidence is recent, concentrated, and consistent.


## Generate All Figures and Export Manifest

This cell regenerates every report figure and writes `report_figure_manifest.csv`. It is safe to run because it only reads final outputs and writes report figures.


In [ ]:
ctx = report.build_all_report_figures(ctx)
manifest = pd.DataFrame([row.__dict__ for row in ctx.manifest_rows])
manifest_path = report.MANIFEST_PATH
print(f"Manifest written to: {manifest_path}")
manifest[["filename", "suggested_report_section", "short_caption"]]


## Final Interpretation

The figures support the full project story: this project is not only a model comparison and not only a few XAI examples. It builds a complete alarm triage workflow. Model evaluation shows whether alarms are detected effectively. Alarm episode grouping turns timestamp predictions into operational events. XAI then adds evidence structure by showing whether alarms are recent-context driven, concentrated, and consistent across methods or models. Together, these outputs support operator judgment under uncertainty.
